In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols, logit, probit
from statsmodels.iolib.summary2 import summary_col
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Загрузка данных
loanapp = pd.read_csv('../datasets/loanapp.csv')
SwissLabor = pd.read_csv('../datasets/SwissLabor.csv')
mroz_Greene = pd.read_csv('../datasets/TableF5-1.csv')

# Предобработка категориальных переменных и удаление пропусков
SwissLabor['participation_num'] = SwissLabor['participation'].replace({'yes': 1, 'no': 0}).astype(int)
SwissLabor['foreign'] = SwissLabor['foreign'].replace({'yes': 1, 'no': 0}).astype(int)

mroz_clean = mroz_Greene[['LFP', 'WA', 'FAMINC', 'WE', 'KL6', 'K618', 'CIT', 'UN']].dropna()
loanapp_clean = loanapp[['approve', 'appinc', 'mortno', 'unem', 'dep', 'male']].dropna()
swiss_clean = SwissLabor[['participation_num', 'income', 'age', 'youngkids', 'oldkids']].dropna()

my_digits = 3

# Функция для удобного вывода предельных значений
def print_margeff(res, at='mean', digits=3):
    """
    Извлекает и печатает таблицу предельных значений.
    at='mean' -> Предельные эффекты в средней точке (MEM)
    at='overall' -> Средние предельные эффекты (AME)
    """
    margeff = res.get_margeff(at=at)
    df_me = margeff.summary_frame().round(digits)
    # Переименовываем столбцы для большей схожести со стандартом
    df_me.columns = ['dy/dx', 'Std. Err.', 'z-value', 'p-value', 'CI Lower', 'CI Upper']
    display(Markdown(df_me.to_markdown()))

legend_text = "\n\n*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$"

# Предельные значения 

## для probit

Рассмотрим probit-модель $P(y=1)=\Phi(x'\beta)$, где
$x'\beta=\beta_0+\beta_1x_1+\dots+\beta_kx_k$

Предельные значения $\frac{\partial P(y=1)}{\partial x_j}=\phi(x'\beta)\beta_j$,
где $\phi(z)=\frac{1}{\sqrt{2\pi}}\exp\{-z^2/2\}$

## для logit

Рассмотрим logit-модель $P(y=1)=\Lambda(x'\beta)$, где
$x'\beta=\beta_0+\beta_1x_1+\dots+\beta_kx_k$

Предельные значения $\frac{\partial P(y=1)}{\partial x_j}=\lambda(x'\beta)\beta_j$,
где $\lambda(z)=\frac{\exp(z)}{(1+\exp(z))^2}$

## Средние предельные значения

Обычно рассчитываются 

- предельные значения для каждого регрессора в средней точке (MEM - Marginal Effects at the Mean): 
  * $\phi(\bar{x}'\beta)\beta_j$ для probit 
  * $\lambda(\bar{x}'\beta)\beta_j$ для logit
- среднее по всей выборке предельное значения для каждого регрессора (AME - Average Marginal Effects): 
  * $\overline{\phi(x'\beta)\beta_j}$  для probit 
  * $\overline{\lambda(x'\beta)\beta_j}$ для logit

# labour force equation

Для датасета `TableF5-1` рассмотрим регрессию __LFP на WA, np.log(FAMINC), WE, KL6, K618, CIT, UN__
трёх спецификаций:

- LPM
- logit
- probit

Результаты подгонки:

In [2]:
#| echo: false
#| warning: false
#| output: asis

f_lfp = 'LFP ~ WA + np.log(FAMINC) + WE + KL6 + K618 + CIT + UN'

regr_LPM_lfp = ols(f_lfp, data=mroz_clean).fit()
regr_logit_lfp = logit(f_lfp, data=mroz_clean).fit(disp=0)
regr_probit_lfp = probit(f_lfp, data=mroz_clean).fit(disp=0)

models_lfp = [regr_LPM_lfp, regr_logit_lfp, regr_probit_lfp]
model_names_lfp = ['LPM', 'logit', 'probit']
order_forecast_lfp = [
    'WA', 'np.log(FAMINC)', 'WE', 'KL6', 'K618', 'CIT', 'UN', 
    'Intercept'
]
metrics_dict = {
    'Nobs': lambda x: f"{int(x.nobs)}",
    'Log-Likelihood': lambda x: f"{x.llf:.{my_digits}f}" if hasattr(x, 'llf') else "-",
    'R-squared': lambda x: f"{x.rsquared:.{my_digits}f}" if hasattr(x, 'rsquared') else "-",
    'Pseudo R-squared': lambda x: f"{x.prsquared:.{my_digits}f}" if hasattr(x, 'prsquared') else "-"
}


compare_table_lfp = summary_col(
    models_lfp, 
    model_names=model_names_lfp,
    stars=True, 
    float_format=f'%0.{my_digits}f',
    regressor_order=order_forecast_lfp,
    info_dict=metrics_dict # Подключаем наши метрики
)

df_comp_lfp = compare_table_lfp.tables[0]

dep_vars_dict_lfp = {col: [m.model.endog_names] for col, m in zip(df_comp_lfp.columns, models_lfp)}
top_row_lfp = pd.DataFrame(dep_vars_dict_lfp, index=['Dep. Variable'])
df_comp_lfp = pd.concat([top_row_lfp, df_comp_lfp])

display(Markdown(df_comp_lfp.to_markdown() + legend_text))

|                  | LPM       | logit     | probit    |
|:-----------------|:----------|:----------|:----------|
| Dep. Variable    | LFP       | LFP       | LFP       |
| WA               | -0.013*** | -0.063*** | -0.038*** |
|                  | (0.003)   | (0.013)   | (0.008)   |
| np.log(FAMINC)   | 0.075**   | 0.341**   | 0.205**   |
|                  | (0.037)   | (0.172)   | (0.104)   |
| WE               | 0.038***  | 0.179***  | 0.108***  |
|                  | (0.008)   | (0.040)   | (0.024)   |
| KL6              | -0.302*** | -1.443*** | -0.868*** |
|                  | (0.036)   | (0.194)   | (0.112)   |
| K618             | -0.018    | -0.095    | -0.057    |
|                  | (0.014)   | (0.067)   | (0.040)   |
| CIT              | -0.048    | -0.214    | -0.126    |
|                  | (0.038)   | (0.176)   | (0.107)   |
| UN               | -0.004    | -0.017    | -0.011    |
|                  | (0.006)   | (0.026)   | (0.016)   |
| Intercept        | 0.079     | -1.856    | -1.108    |
|                  | (0.356)   | (1.679)   | (1.014)   |
| R-squared        | 0.130     |           |           |
| R-squared Adj.   | 0.122     |           |           |
| Log-Likelihood   | -487.089  | -462.420  | -462.554  |
| Nobs             | 753       | 753       | 753       |
| Pseudo R-squared | -         | 0.102     | 0.102     |
| R-squared        | 0.130     | -         | -         |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [3]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_lfp, at='mean', digits=my_digits)

|                |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:---------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| WA             |  -0.015 |       0.003 |    -5.006 |     0     |     -0.021 |     -0.009 |
| np.log(FAMINC) |   0.083 |       0.042 |     1.982 |     0.048 |      0.001 |      0.166 |
| WE             |   0.044 |       0.01  |     4.45  |     0     |      0.025 |      0.063 |
| KL6            |  -0.353 |       0.048 |    -7.395 |     0     |     -0.446 |     -0.259 |
| K618           |  -0.023 |       0.016 |    -1.416 |     0.157 |     -0.055 |      0.009 |
| CIT            |  -0.052 |       0.043 |    -1.211 |     0.226 |     -0.137 |      0.032 |
| UN             |  -0.004 |       0.006 |    -0.675 |     0.5   |     -0.017 |      0.008 |

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [4]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_lfp, at='mean', digits=my_digits)

|                |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:---------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| WA             |  -0.015 |       0.003 |    -5.053 |     0     |     -0.021 |     -0.009 |
| np.log(FAMINC) |   0.081 |       0.041 |     1.973 |     0.049 |      0.001 |      0.161 |
| WE             |   0.042 |       0.009 |     4.504 |     0     |      0.024 |      0.061 |
| KL6            |  -0.34  |       0.044 |    -7.738 |     0     |     -0.427 |     -0.254 |
| K618           |  -0.022 |       0.016 |    -1.407 |     0.16  |     -0.053 |      0.009 |
| CIT            |  -0.049 |       0.042 |    -1.17  |     0.242 |     -0.132 |      0.033 |
| UN             |  -0.004 |       0.006 |    -0.67  |     0.503 |     -0.016 |      0.008 |

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [5]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_lfp, at='overall', digits=my_digits)

|                |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:---------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| WA             |  -0.013 |       0.003 |    -5.324 |     0     |     -0.018 |     -0.008 |
| np.log(FAMINC) |   0.073 |       0.036 |     2.001 |     0.045 |      0.001 |      0.144 |
| WE             |   0.038 |       0.008 |     4.667 |     0     |      0.022 |      0.054 |
| KL6            |  -0.307 |       0.036 |    -8.642 |     0     |     -0.377 |     -0.238 |
| K618           |  -0.02  |       0.014 |    -1.422 |     0.155 |     -0.048 |      0.008 |
| CIT            |  -0.046 |       0.037 |    -1.215 |     0.224 |     -0.119 |      0.028 |
| UN             |  -0.004 |       0.005 |    -0.676 |     0.499 |     -0.014 |      0.007 |

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [6]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_lfp, at='overall', digits=my_digits)

|                |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:---------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| WA             |  -0.013 |       0.003 |    -5.314 |     0     |     -0.018 |     -0.008 |
| np.log(FAMINC) |   0.072 |       0.036 |     1.987 |     0.047 |      0.001 |      0.143 |
| WE             |   0.038 |       0.008 |     4.686 |     0     |      0.022 |      0.054 |
| KL6            |  -0.304 |       0.035 |    -8.818 |     0     |     -0.372 |     -0.237 |
| K618           |  -0.02  |       0.014 |    -1.412 |     0.158 |     -0.048 |      0.008 |
| CIT            |  -0.044 |       0.038 |    -1.173 |     0.241 |     -0.118 |      0.03  |
| UN             |  -0.004 |       0.005 |    -0.671 |     0.502 |     -0.014 |      0.007 |

# approve equation

Для датасета `loanapp` рассмотрим регрессию __approve на I(appinc / 100), mortno, unem, dep, male__
трёх спецификаций:

- LPM
- logit
- probit

Результаты подгонки:

In [7]:
#| echo: false
#| warning: false
#| output: asis

f_app = 'approve ~ I(appinc/100) + mortno + unem + dep + male'

regr_LPM_app = ols(f_app, data=loanapp_clean).fit()
regr_logit_app = logit(f_app, data=loanapp_clean).fit(disp=0)
regr_probit_app = probit(f_app, data=loanapp_clean).fit(disp=0)

models_app = [regr_LPM_app, regr_logit_app, regr_probit_app]
model_names_app = ['LPM', 'logit', 'probit']
order_forecast_app = [
    'I(appinc / 100)', 'mortno', 'unem', 'dep', 'male', 
    'Intercept'
]
metrics_dict = {
    'Nobs': lambda x: f"{int(x.nobs)}",
    'Log-Likelihood': lambda x: f"{x.llf:.{my_digits}f}" if hasattr(x, 'llf') else "-",
    'R-squared': lambda x: f"{x.rsquared:.{my_digits}f}" if hasattr(x, 'rsquared') else "-",
    'Pseudo R-squared': lambda x: f"{x.prsquared:.{my_digits}f}" if hasattr(x, 'prsquared') else "-"
}

compare_table_app = summary_col(
    models_app, 
    model_names=model_names_app,
    stars=True, 
    float_format=f'%0.{my_digits}f',
    regressor_order=order_forecast_app,
    info_dict=metrics_dict # Подключаем наши метрики
)

df_comp_app = compare_table_app.tables[0]

dep_vars_dict_app = {col: [m.model.endog_names] for col, m in zip(df_comp_app.columns, models_app)}
top_row_app = pd.DataFrame(dep_vars_dict_app, index=['Dep. Variable'])
df_comp_app = pd.concat([top_row_app, df_comp_app])

display(Markdown(df_comp_app.to_markdown() + legend_text))

|                  | LPM      | logit    | probit   |
|:-----------------|:---------|:---------|:---------|
| Dep. Variable    | approve  | approve  | approve  |
| I(appinc / 100)  | -0.014*  | -0.106   | -0.056   |
|                  | (0.009)  | (0.067)  | (0.038)  |
| mortno           | 0.079*** | 0.817*** | 0.422*** |
|                  | (0.016)  | (0.170)  | (0.086)  |
| unem             | -0.008** | -0.065** | -0.036** |
|                  | (0.003)  | (0.029)  | (0.016)  |
| dep              | -0.012*  | -0.106*  | -0.055*  |
|                  | (0.007)  | (0.061)  | (0.033)  |
| male             | 0.019    | 0.173    | 0.093    |
|                  | (0.019)  | (0.176)  | (0.094)  |
| Intercept        | 0.886*** | 2.032*** | 1.198*** |
|                  | (0.022)  | (0.193)  | (0.105)  |
| R-squared        | 0.017    |          |          |
| R-squared Adj.   | 0.014    |          |          |
| Log-Likelihood   | -590.863 | -720.861 | -720.982 |
| Nobs             | 1971     | 1971     | 1971     |
| Pseudo R-squared | -        | 0.023    | 0.023    |
| R-squared        | 0.017    | -        | -        |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [8]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_app, at='mean', digits=my_digits)

|                 |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:----------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| I(appinc / 100) |  -0.011 |       0.007 |    -1.583 |     0.114 |     -0.025 |      0.003 |
| mortno          |   0.084 |       0.017 |     5     |     0     |      0.051 |      0.118 |
| unem            |  -0.007 |       0.003 |    -2.26  |     0.024 |     -0.013 |     -0.001 |
| dep             |  -0.011 |       0.006 |    -1.741 |     0.082 |     -0.023 |      0.001 |
| male            |   0.018 |       0.018 |     0.985 |     0.325 |     -0.018 |      0.054 |

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [9]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_app, at='mean', digits=my_digits)

|                 |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:----------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| I(appinc / 100) |  -0.011 |       0.008 |    -1.487 |     0.137 |     -0.026 |      0.004 |
| mortno          |   0.084 |       0.017 |     5.014 |     0     |      0.051 |      0.116 |
| unem            |  -0.007 |       0.003 |    -2.281 |     0.023 |     -0.013 |     -0.001 |
| dep             |  -0.011 |       0.007 |    -1.65  |     0.099 |     -0.024 |      0.002 |
| male            |   0.018 |       0.019 |     0.987 |     0.324 |     -0.018 |      0.055 |

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [10]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_app, at='overall', digits=my_digits)

|                 |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:----------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| I(appinc / 100) |  -0.011 |       0.007 |    -1.581 |     0.114 |     -0.025 |      0.003 |
| mortno          |   0.087 |       0.018 |     4.764 |     0     |      0.051 |      0.123 |
| unem            |  -0.007 |       0.003 |    -2.251 |     0.024 |     -0.013 |     -0.001 |
| dep             |  -0.011 |       0.007 |    -1.736 |     0.083 |     -0.024 |      0.001 |
| male            |   0.018 |       0.019 |     0.984 |     0.325 |     -0.018 |      0.055 |

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [11]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_app, at='overall', digits=my_digits)

|                 |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:----------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| I(appinc / 100) |  -0.011 |       0.008 |    -1.488 |     0.137 |     -0.026 |      0.004 |
| mortno          |   0.084 |       0.017 |     4.931 |     0     |      0.051 |      0.118 |
| unem            |  -0.007 |       0.003 |    -2.28  |     0.023 |     -0.013 |     -0.001 |
| dep             |  -0.011 |       0.007 |    -1.649 |     0.099 |     -0.024 |      0.002 |
| male            |   0.019 |       0.019 |     0.987 |     0.324 |     -0.018 |      0.056 |

# swiss labour force equation

Для датасета `SwissLabour` рассмотрим регрессию __participation на income, age, I(age ** 2), youngkids, oldkids__
трёх спецификаций:

- LPM
- logit
- probit

Результаты подгонки:

In [12]:
#| echo: false
#| warning: false
#| output: asis

f_swiss = 'participation_num ~ income + age + I(age**2) + youngkids + oldkids'

regr_LPM_swiss = ols(f_swiss, data=swiss_clean).fit()
regr_logit_swiss = logit(f_swiss, data=swiss_clean).fit(disp=0)
regr_probit_swiss = probit(f_swiss, data=swiss_clean).fit(disp=0)

models_swiss = [regr_LPM_swiss, regr_logit_swiss, regr_probit_swiss]
model_names_swiss = ['LPM', 'logit', 'probit']
order_forecast_swiss = [
    'income', 'I(income ** 2)', 'age', 'I(age ** 2)', 'youngkids', 'oldkids', 
    'Intercept'
]
metrics_dict = {
    'Nobs': lambda x: f"{int(x.nobs)}",
    'Log-Likelihood': lambda x: f"{x.llf:.{my_digits}f}" if hasattr(x, 'llf') else "-",
    'R-squared': lambda x: f"{x.rsquared:.{my_digits}f}" if hasattr(x, 'rsquared') else "-",
    'Pseudo R-squared': lambda x: f"{x.prsquared:.{my_digits}f}" if hasattr(x, 'prsquared') else "-"
}

compare_table_swiss = summary_col(
    models_swiss, 
    model_names=model_names_swiss,
    stars=True, 
    float_format=f'%0.{my_digits}f',
    regressor_order=order_forecast_swiss,
    info_dict=metrics_dict # Подключаем наши метрики
)

df_comp_swiss = compare_table_swiss.tables[0]

dep_vars_dict_swiss = {col: [m.model.endog_names] for col, m in zip(df_comp_swiss.columns, models_swiss)}
top_row_swiss = pd.DataFrame(dep_vars_dict_swiss, index=['Dep. Variable'])
df_comp_swiss = pd.concat([top_row_swiss, df_comp_swiss])

display(Markdown(df_comp_swiss.to_markdown() + legend_text))

|                  | LPM               | logit             | probit            |
|:-----------------|:------------------|:------------------|:------------------|
| Dep. Variable    | participation_num | participation_num | participation_num |
| income           | -0.258***         | -1.337***         | -0.805***         |
|                  | (0.039)           | (0.214)           | (0.125)           |
| age              | 0.795***          | 3.907***          | 2.361***          |
|                  | (0.131)           | (0.670)           | (0.397)           |
| I(age ** 2)      | -0.112***         | -0.550***         | -0.332***         |
|                  | (0.016)           | (0.083)           | (0.049)           |
| youngkids        | -0.228***         | -1.069***         | -0.640***         |
|                  | (0.032)           | (0.166)           | (0.096)           |
| oldkids          | -0.053***         | -0.252***         | -0.152***         |
|                  | (0.017)           | (0.083)           | (0.050)           |
| Intercept        | 2.082***          | 8.431***          | 5.055***          |
|                  | (0.446)           | (2.330)           | (1.388)           |
| R-squared        | 0.155             |                   |                   |
| R-squared Adj.   | 0.150             |                   |                   |
| Log-Likelihood   | -556.793          | -527.064          | -527.405          |
| Nobs             | 872               | 872               | 872               |
| Pseudo R-squared | -                 | 0.124             | 0.123             |
| R-squared        | 0.155             | -                 | -                 |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [13]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_swiss, at='mean', digits=my_digits)

|             |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| income      |  -0.33  |       0.053 |    -6.249 |     0     |     -0.434 |     -0.227 |
| age         |   0.965 |       0.165 |     5.849 |     0     |      0.642 |      1.289 |
| I(age ** 2) |  -0.136 |       0.02  |    -6.633 |     0     |     -0.176 |     -0.096 |
| youngkids   |  -0.264 |       0.041 |    -6.458 |     0     |     -0.344 |     -0.184 |
| oldkids     |  -0.062 |       0.02  |    -3.052 |     0.002 |     -0.102 |     -0.022 |

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [14]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_swiss, at='mean', digits=my_digits)

|             |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| income      |  -0.318 |       0.05  |    -6.428 |     0     |     -0.415 |     -0.221 |
| age         |   0.934 |       0.157 |     5.95  |     0     |      0.627 |      1.242 |
| I(age ** 2) |  -0.131 |       0.019 |    -6.79  |     0     |     -0.169 |     -0.093 |
| youngkids   |  -0.253 |       0.038 |    -6.69  |     0     |     -0.327 |     -0.179 |
| oldkids     |  -0.06  |       0.02  |    -3.055 |     0.002 |     -0.099 |     -0.022 |

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [15]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_swiss, at='overall', digits=my_digits)

|             |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| income      |  -0.279 |       0.041 |    -6.788 |     0     |     -0.36  |     -0.199 |
| age         |   0.817 |       0.13  |     6.288 |     0     |      0.562 |      1.071 |
| I(age ** 2) |  -0.115 |       0.016 |    -7.295 |     0     |     -0.146 |     -0.084 |
| youngkids   |  -0.224 |       0.032 |    -7.082 |     0     |     -0.285 |     -0.162 |
| oldkids     |  -0.053 |       0.017 |    -3.112 |     0.002 |     -0.086 |     -0.019 |

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [16]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_swiss, at='overall', digits=my_digits)

|             |   dy/dx |   Std. Err. |   z-value |   p-value |   CI Lower |   CI Upper |
|:------------|--------:|------------:|----------:|----------:|-----------:|-----------:|
| income      |  -0.278 |       0.04  |    -6.898 |     0     |     -0.357 |     -0.199 |
| age         |   0.815 |       0.129 |     6.323 |     0     |      0.562 |      1.067 |
| I(age ** 2) |  -0.115 |       0.016 |    -7.359 |     0     |     -0.145 |     -0.084 |
| youngkids   |  -0.221 |       0.031 |    -7.232 |     0     |     -0.281 |     -0.161 |
| oldkids     |  -0.053 |       0.017 |    -3.104 |     0.002 |     -0.086 |     -0.019 |